# State Migration - 04: Recovering old checkpoints

> **MLCourse - Agentic AI - LangGraph - Module 10**

Notebook 03 built **lazy** migration: a thread is upgraded when someone next
touches it. That covers most threads and costs nothing until it is needed.

It leaves two holes, and this notebook fills them.

1. **Dormant threads.** A thread nobody touches keeps its old state forever.
   Your migration code can never be deleted, because there might still be a v1
   thread out there. Two years later you are maintaining a five-step chain for
   threads that may not exist.
2. **Suspended threads.** A thread paused at an `interrupt()` resumes *into
   the node that was waiting* - not at `START`. Your migration node at the
   entry point **never runs**, so lazy migration cannot reach it at all.

### What you will learn

1. Enumerating every thread in a checkpoint store.
2. Writing an **eager backfill** that migrates all of them in bulk.
3. Using `update_state()` to repair a thread from outside the graph.
4. Why suspended threads need this and cannot be fixed lazily - demonstrated.
5. Verifying a backfill, and the safety rules for running one.

Still no LLM calls and no API key.

### Setup: rebuild the scenario from notebook 03


In [ ]:
import os
import sqlite3
from typing import NotRequired, TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

DB_PATH = "recovery_demo.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

checkpointer = SqliteSaver(sqlite3.connect(DB_PATH, check_same_thread=False))
CURRENT_SCHEMA_VERSION = 3


# --- the migration chain from notebook 03, unchanged ---
def migrate_v1_to_v2(state: dict) -> dict:
    return {**state, "schema_version": 2,
            "priority": state.get("priority", "normal")}


def migrate_v2_to_v3(state: dict) -> dict:
    migrated = {**state, "schema_version": 3,
                "body": state.get("body") or state.get("text", ""),
                "team": state.get("team", "unassigned")}
    migrated.pop("text", None)
    return migrated


MIGRATIONS = {1: migrate_v1_to_v2, 2: migrate_v2_to_v3}


def migrate(state: dict) -> dict:
    version = state.get("schema_version", 1)
    while version < CURRENT_SCHEMA_VERSION:
        step = MIGRATIONS.get(version)
        if step is None:
            raise RuntimeError(f"no migration from v{version}")
        state = step(state)
        if state["schema_version"] <= version:
            raise RuntimeError("migration did not advance the version")
        version = state["schema_version"]
    return state


print("migration chain ready; checkpoint store is fresh")


### Seed a realistic store: threads at several schema generations


In [ ]:
class TicketStateV1(TypedDict):
    ticket_id: str
    text: str


class TicketStateV2(TypedDict):
    schema_version: int
    ticket_id: str
    text: str
    priority: str


def old_work(state) -> dict:
    return {"text": state["text"] + " [handled]"}


def _compile(cls):
    b = StateGraph(cls)
    b.add_node("work", old_work)
    b.add_edge(START, "work")
    b.add_edge("work", END)
    return b.compile(checkpointer=checkpointer)


legacy_v1, legacy_v2 = _compile(TicketStateV1), _compile(TicketStateV2)

# Four v1 threads and two v2 threads, as a real store would have.
for i in range(4):
    legacy_v1.invoke({"ticket_id": f"T-{i}", "text": f"issue {i}"},
                     {"configurable": {"thread_id": f"old-{i}"}})
for i in range(2):
    legacy_v2.invoke({"schema_version": 2, "ticket_id": f"T-1{i}",
                      "text": f"issue 1{i}", "priority": "high"},
                     {"configurable": {"thread_id": f"mid-{i}"}})

print("seeded 4 v1 threads and 2 v2 threads")


### 1. Enumerating every thread

`checkpointer.list(None)` walks the entire store rather than one thread. It
yields a `CheckpointTuple` **per checkpoint**, and a thread has many - one per
superstep - so the first job is to reduce that to the *latest* checkpoint per
thread.

### Finding every thread and its current schema version


In [ ]:
def all_threads(cp) -> dict:
    """Map thread_id -> its most recent CheckpointTuple.

    list() yields newest-first, so the FIRST tuple seen for a thread is its
    latest checkpoint and later ones can be skipped.
    """
    latest = {}
    for t in cp.list(None):
        tid = t.config["configurable"]["thread_id"]
        if tid not in latest:
            latest[tid] = t
    return latest


threads = all_threads(checkpointer)
print(f"{len(threads)} threads in the store\n")
print(f"{'thread':<10} {'version':>8}  channel_values")
print("-" * 76)
for tid, t in sorted(threads.items()):
    values = t.checkpoint["channel_values"]
    print(f"{tid:<10} {values.get('schema_version', 1):>8}  {values}")


Note that we are reading `checkpoint["channel_values"]` **directly from the
checkpointer**, not through `graph.get_state()`.

That distinction matters. `get_state()` filters to the channels the current
graph declares and replays writes through the current reducers - which is
exactly what hid `text` in notebook 03, and exactly what *crashed* on the
reducer change in notebook 02, Case D. The raw checkpointer has no such
opinions: it gives you what was stored.

**For migration tooling, always read raw.** It is the only view that shows you
what you actually have to work with.

### 2. The eager backfill

Now upgrade every thread in place. The mechanism is `graph.update_state()`,
which writes a new checkpoint containing your changes - the same API
`04_human_in_the_loop` uses to edit state at a breakpoint, here pointed at
schema repair.

The backfill needs the **transitional** state class from notebook 03, for the
same two reasons: the graph must declare the legacy channel so it is loaded,
and the node signature must not filter it out.

### The current (transitional) graph


In [ ]:
class TicketStateV3Migrating(TypedDict):
    schema_version: int
    ticket_id: str
    body: str
    priority: str
    team: str
    text: NotRequired[str]      # DEPRECATED, kept for the migration window


def triage(state: TicketStateV3Migrating) -> dict:
    return {"team": "escalations" if state["priority"] == "high" else state["team"]}


b3 = StateGraph(TicketStateV3Migrating)
b3.add_node("triage", triage)
b3.add_edge(START, "triage")
b3.add_edge("triage", END)
agent_v3 = b3.compile(checkpointer=checkpointer)

print("current graph compiled (transitional schema, no migration node --")
print("the backfill below replaces lazy migration entirely)")


### The backfill


In [ ]:
def backfill(cp, graph, dry_run: bool = True) -> dict:
    """Migrate every thread in the store to CURRENT_SCHEMA_VERSION.

    dry_run=True reports what WOULD change without writing anything. Always
    run it that way first -- a migration bug applied to every thread at once
    is a very bad afternoon.
    """
    stats = {"scanned": 0, "migrated": 0, "already_current": 0, "failed": 0}
    failures = []

    for tid, tup in sorted(all_threads(cp).items()):
        stats["scanned"] += 1
        stored = dict(tup.checkpoint["channel_values"])
        # Internal bookkeeping channels are not part of your state.
        stored.pop("__start__", None)

        before = stored.get("schema_version", 1)
        if before >= CURRENT_SCHEMA_VERSION:
            stats["already_current"] += 1
            print(f"  {tid:<10} v{before} already current")
            continue

        try:
            upgraded = migrate(stored)
        except Exception as e:                    # one bad thread must not
            stats["failed"] += 1                  # abort the whole backfill
            failures.append((tid, repr(e)))
            print(f"  {tid:<10} FAILED: {e}")
            continue

        changes = {k: v for k, v in upgraded.items() if stored.get(k) != v}
        if dry_run:
            print(f"  {tid:<10} v{before} -> v{upgraded['schema_version']} "
                  f"WOULD SET {changes}")
        else:
            graph.update_state({"configurable": {"thread_id": tid}}, changes)
            print(f"  {tid:<10} v{before} -> v{upgraded['schema_version']} migrated")
        stats["migrated"] += 1

    stats["failures"] = failures
    return stats


print("=== DRY RUN ===")
dry = backfill(checkpointer, agent_v3, dry_run=True)
print(f"\n{dry}")


### Now for real


In [ ]:
print("=== APPLYING ===")
applied = backfill(checkpointer, agent_v3, dry_run=False)
print(f"\nscanned={applied['scanned']} migrated={applied['migrated']} "
      f"already_current={applied['already_current']} failed={applied['failed']}")


### Verification: re-scan and confirm every thread is current


In [ ]:
# A backfill you did not verify is a backfill you did not do.

print(f"{'thread':<10} {'version':>8} {'body':<34} {'text left?':>11}")
print("-" * 68)
stale = []
for tid, t in sorted(all_threads(checkpointer).items()):
    v = t.checkpoint["channel_values"]
    ver = v.get("schema_version", 1)
    if ver < CURRENT_SCHEMA_VERSION:
        stale.append(tid)
    print(f"{tid:<10} {ver:>8} {str(v.get('body', ''))[:32]:<34} "
          f"{'yes' if 'text' in v else 'no':>11}")

print(f"\nthreads still below v{CURRENT_SCHEMA_VERSION}: {stale or 'none'}")
print("\nEvery 'body' is populated -- the rename survived, in bulk.")
print("'text' still lingers: update_state() cannot DELETE a channel, only")
print("write to one. Dropping it is the 'contract' step -- remove it from the")
print("state class in a later release, once nothing reads it.")


### What the backfill bought you

- Every thread is at v3 **now**, not whenever a user happens to return.
- `MIGRATIONS[1]` and `MIGRATIONS[2]` can be deleted in a future release,
  because the verification scan proves no thread needs them.
- The one-off cost is bounded and happens at a time you chose, rather than
  being spread across production traffic.

Note the honest limitation printed above: `update_state()` writes channels, it
does not remove them. `text` is still on every thread. That is fine - it is
what the **contract** step is for - but it means "the backfill is done" and
"the old field is gone" are two different milestones.

### 3. The threads lazy migration cannot reach

Now the case that motivates all of this. A thread suspended at an
`interrupt()` **resumes into the waiting node**, not at `START`. A migration
node at the graph entry is simply not on the resume path.

Let's build one and watch lazy migration fail to help.

### A thread suspended under the OLD schema


In [ ]:
class ReviewV1(TypedDict):
    document: str
    decision: str


def review_v1(state: ReviewV1) -> dict:
    return {"decision": interrupt({"question": f"Approve {state['document']}?"})}


rb1 = StateGraph(ReviewV1)
rb1.add_node("review", review_v1)
rb1.add_edge(START, "review")
rb1.add_edge("review", END)
review_agent_v1 = rb1.compile(checkpointer=checkpointer)

rcfg = {"configurable": {"thread_id": "review-suspended"}}
review_agent_v1.invoke({"document": "contract.pdf", "decision": ""}, rcfg)

snap = review_agent_v1.get_state(rcfg)
print("FRIDAY: thread suspended awaiting human approval")
print(f"  values : {snap.values}")
print(f"  next   : {snap.next}   <- the pending task")


### The new schema ships, WITH a migration node at START


In [ ]:
class ReviewV2(TypedDict):
    schema_version: NotRequired[int]
    document: str
    decision: str
    reviewer: str               # NEW, and the review node now reads it


def migrate_review(state: ReviewV2) -> dict:
    """A lazy migration node at the graph entry -- exactly as notebook 03
    recommends. Watch it not run."""
    if state.get("schema_version") == 2:
        return {}
    print("    [migration node] ran -- setting reviewer default")
    return {"schema_version": 2, "reviewer": state.get("reviewer", "unknown")}


def review_v2(state: ReviewV2) -> dict:
    answer = interrupt({"question": f"Approve {state['document']}?"})
    return {"decision": f"{answer} by {state['reviewer']}"}


rb2 = StateGraph(ReviewV2)
rb2.add_node("migrate", migrate_review)
rb2.add_node("review", review_v2)
rb2.add_edge(START, "migrate")
rb2.add_edge("migrate", "review")
rb2.add_edge("review", END)
review_agent_v2 = rb2.compile(checkpointer=checkpointer)

print("MONDAY: the human approves. Resuming under the new schema, WITHOUT")
print("repairing state first (the naive thing to try):")
try:
    review_agent_v2.invoke(Command(resume="approved"), rcfg)
except KeyError as e:
    print(f"  KeyError: {e}")

after_crash = review_agent_v2.get_state(rcfg)
print(f"\n  state is now: {after_crash.values}")
print(f"  next task   : {after_crash.next}")
print("\n  Worse than a clean failure: the interrupt was CONSUMED (the node")
print("  received 'approved' before it crashed), so 'next' is now EMPTY.")
print("  The thread is neither pending nor complete. This attempt cannot be")
print("  repeated -- the human's answer is already spent, and it went nowhere.")


### This is worse than notebook 01 described, and it is the hole eager
### migration exists to fill

`Command(resume=...)` re-enters the interrupted node directly. It never
passes through `START`, so the migration node - correct, installed,
irrelevant - never ran.

But look closely at what the crash actually did: `interrupt()` returned
`"approved"` to the node, the node then crashed reading `state['reviewer']`
**after** consuming the human's answer, and the checkpoint was left with no
pending task at all. The thread is not stuck waiting - it is stuck *broken*,
and the same resume cannot be attempted again because there is no interrupt
left to resume.

The lesson: **never attempt a resume on a thread whose schema may have
changed. Repair first, always.** Let's do a second suspended thread properly.

### Doing it right: repair BEFORE ever attempting resume


In [ ]:
rcfg2 = {"configurable": {"thread_id": "review-suspended-2"}}
review_agent_v1.invoke({"document": "invoice.pdf", "decision": ""}, rcfg2)

before = review_agent_v2.get_state(rcfg2)
print(f"BEFORE repair: values={before.values}  next={before.next}")

# THE IMPORTANT DETAIL: as_node="migrate". Without it, update_state() treats
# the write as a brand-new entry and resets 'next' to the graph's START edge
# -- which would re-run 'migrate' AND 'review' from scratch, asking the human
# the question a second time. as_node="migrate" instead says "pretend the
# migration node already ran and produced these values", which preserves the
# ORIGINAL pending task ('review') untouched.
review_agent_v2.update_state(
    rcfg2, {"schema_version": 2, "reviewer": "unknown"}, as_node="migrate")

after = review_agent_v2.get_state(rcfg2)
print(f"\nAFTER update_state(as_node='migrate'): values={after.values}")
print(f"  next: {after.next}   <- STILL ('review',). The interrupt is untouched.")

print("\nNow resume for the first and only time:")
result = review_agent_v2.invoke(Command(resume="approved"), rcfg2)
print(f"  {result}")


### Why `as_node` is the detail that makes this work

`update_state()` needs to be told **which node's output this looks like**, so
it can compute the correct next pending task from the graph's edges.

- No `as_node` (or `as_node="review"`) → LangGraph computes `next` as if that
  node just finished, which for `"review"` means *the graph is complete* -
  and for no `as_node` at all, it falls back to treating the write as a fresh
  entry, which recomputes `next` from `START` and would run `migrate` *and*
  `review` again, asking the human a second time.
- `as_node="migrate"` → tells LangGraph the write looks like the migration
  node's output, so `next` becomes whatever comes **after** `migrate` in the
  graph - which is `review`, exactly the task that was already pending. The
  interrupt is untouched because nothing about it was rewritten.

The thread completed, and `decision` records **the approval the human gave on
Friday**, attributed to the migrated `reviewer` default. Nobody was asked
twice - but only because we picked the node name that matches what the
migration was standing in for, and only because we repaired *before* ever
calling `Command(resume=...)`.

### 4. Lazy vs eager: which to use

They are not alternatives - most systems want both.

| | **Lazy** (migration node) | **Eager** (backfill script) |
|---|---|---|
| Runs when | a thread is next used | you run it |
| Cost | spread over normal traffic | one bounded batch |
| Dormant threads | never migrated | migrated |
| **Suspended threads** | **never migrated** | migrated |
| Lets you delete old migrations? | no - a v1 thread might still exist | yes, after verification |
| Risk | low, incremental | a bug hits every thread at once |
| Needs a deploy? | yes | no - a script against the store |

**Use both, in this order:**

1. Ship the **lazy** migration node first. It is safe, incremental, and
   immediately stops new breakage.
2. Run the **eager** backfill once the migration has proven itself in
   production.
3. **Verify** by re-scanning for anything below the current version.
4. Only then delete the old migration steps and **contract** the schema.

### Safety rules for a backfill

- **Dry-run first.** Print what would change; read it before writing.
- **Back up the store.** For SQLite that is copying one file. There is no
  undo.
- **Never let one bad thread abort the batch** - catch per thread, collect
  failures, and report them (our `backfill` does this).
- **Make it re-runnable.** It is idempotent for the same reason the migration
  chain is: a thread already at the current version is skipped.
- **Check for suspended threads before deploying**, not after. `snapshot.next`
  being non-empty is the signal.

### Key takeaways

- Read migration tooling's input **raw from the checkpointer**, not via
  `get_state()` - `get_state()` filters to current channels and replays
  through current reducers, which is what hides (or crashes on) legacy data.
- `checkpointer.list(None)` walks every thread; take the **first** tuple per
  thread to get its latest checkpoint.
- `graph.update_state()` writes state **without running any node**, which is
  what makes it safe for repairing threads - including suspended ones.
- **Lazy migration cannot reach a suspended thread.** Measured above:
  `Command(resume=...)` re-enters the pending node directly and skips `START`
  entirely, so the migration node never ran.
- **Never attempt a resume on a possibly-stale thread before repairing it.**
  Measured above: the failed resume did not fail cleanly - `interrupt()`
  handed the node `"approved"`, the node then crashed reading the missing
  field, and the checkpoint was left with `next=()`: neither pending nor
  complete, and un-retryable, because the human's answer was already
  consumed. Repair first, always.
- **`as_node` controls which pending task survives an `update_state()` call.**
  Measured above: repairing with `as_node="migrate"` (the node the write
  stands in for) preserved `next=('review',)` and the original interrupt;
  omitting it reset `next` to `START`'s edge, which would have asked the human
  the question again.
- `update_state()` **cannot delete a channel**. Removing a deprecated field is
  the separate **contract** step.
- Always **dry-run**, back up, tolerate per-thread failures, and **verify** by
  re-scanning afterwards.

That completes module 10. Back to the
[module README](README.md), or on to the rest of the
[LangGraph track](../README.md).

Related reading:
[`03_persistence_checkpointing`](../03_persistence_checkpointing/README.md)
for where checkpoints come from,
[`04_human_in_the_loop`](../04_human_in_the_loop/README.md) for `interrupt()`
and `update_state()`, and
`06_agent_patterns/14_async_human_approval` for approvals that stay suspended
long enough to make everything in this module matter.